# APL Logistics – Delivery Performance & SLA Analysis

## Objective
To evaluate shipment delivery performance, measure SLA compliance, identify operational inefficiencies, and assess the financial impact of delivery delays.

## Business Context
APL Logistics handles over 180,000 shipments. A high late-delivery rate has been observed, and management requires structured diagnostics to determine the root cause and strategic improvement areas.

## 1. Data Loading & Validation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("APL_Logistics.csv", encoding="latin1")

df.shape
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Fname                 180519 non-null  object 
 12  Customer Id                   

Type                             0
Days for shipping (real)         0
Days for shipment (scheduled)    0
Benefit per order                0
Sales per customer               0
Delivery Status                  0
Late_delivery_risk               0
Category Id                      0
Category Name                    0
Customer City                    0
Customer Country                 0
Customer Fname                   0
Customer Id                      0
Customer Lname                   8
Customer Segment                 0
Customer State                   0
Customer Street                  0
Customer Zipcode                 3
Department Id                    0
Department Name                  0
Latitude                         0
Longitude                        0
Market                           0
Order City                       0
Order Country                    0
Order Customer Id                0
Order Item Discount              0
Order Item Discount Rate         0
Order Item Product P

### Observation
- Total Orders: 180,519
- Features: 40
- Data completeness > 99%
- No major structural data quality issues

## 2. Delivery Gap Engineering

Delivery Gap = Actual Shipping Days - Scheduled Shipping Days

Classification:
- Delayed → gap > 0
- On-Time → gap = 0
- Early → gap < 0

In [3]:
df["delivery_gap"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]

df["delivery_status"] = np.where(
    df["delivery_gap"] > 0, "Delayed",
    np.where(df["delivery_gap"] == 0, "On-Time", "Early")
)

(df["delivery_status"].value_counts(normalize=True) * 100).round(2)

delivery_status
Delayed    57.28
Early      24.02
On-Time    18.70
Name: proportion, dtype: float64

### Key Finding
57% of shipments are classified as delayed.
Most delays are 1-day in severity, indicating systematic scheduling misalignment rather than extreme operational failure.

## 3. Shipping Mode Performance Analysis

In [4]:
pd.crosstab(df["Shipping Mode"], df["delivery_status"], normalize="index") * 100

delivery_status,Delayed,Early,On-Time
Shipping Mode,,,
First Class,100.000000,0.000000,0.000000
Same Day,47.827873,0.000000,52.172127
Second Class,79.730804,0.000000,20.269196
Standard Class,39.768171,40.246121,19.985708


In [5]:
df.groupby("Shipping Mode")[["Days for shipment (scheduled)", "Days for shipping (real)"]].mean()

,Days for shipment (scheduled),Days for shipping (real)
Shipping Mode,,
First Class,1.0,2.000000
Same Day,0.0,0.478279
Second Class,2.0,3.990828
Standard Class,4.0,3.995907


### Key Insight
Premium shipping modes (First Class, Second Class) demonstrate high delay rates due to unrealistic scheduling commitments.

First Class is scheduled for 1 day but consistently requires 2 days on average, resulting in systemic delay classification.

Standard Class shows near-perfect alignment between scheduled and actual delivery times.

## 4. Regional Delay Analysis

In [6]:
region_delay = pd.crosstab(
    df["Order Region"],
    df["delivery_status"],
    normalize="index"
) * 100

region_delay.sort_values(by="Delayed", ascending=False).head(10)

delivery_status,Delayed,Early,On-Time
Order Region,,,
Central Africa,60.703637,21.407275,17.889088
Western Europe,58.515622,23.615773,17.868605
South Asia,58.504721,23.658000,17.837278
South of USA,58.096415,24.103832,17.799753
Southeast Asia,57.983017,24.122025,17.894958
East of USA,57.975416,23.528561,18.496023
West Asia,57.497088,22.982193,19.520719
East Africa,57.451404,22.786177,19.762419
Eastern Europe,57.448980,23.290816,19.260204


### Key Insight
Delay rates are relatively uniform across regions (57–60%), suggesting systemic scheduling issues rather than geographic bottlenecks.

## 5. Financial Impact Assessment

In [8]:
df.groupby("delivery_status")["Order Profit Per Order"].mean()
df.groupby("delivery_status")["Sales"].mean()
df.groupby("delivery_status")["Order Item Discount Rate"].mean()

delivery_status
Delayed    0.101729
Early      0.101815
On-Time    0.101294
Name: Order Item Discount Rate, dtype: float64

### Key Insight
Delayed shipments generate slightly lower profit per order (~0.95 units lower).
While per-order impact is modest, cumulative effect across 103,000 delayed shipments is financially meaningful.

## 6. Executive Summary

1. Overall delay rate is 57%.
2. Majority of delays are limited to 1 day.
3. Premium shipping modes suffer from SLA misalignment.
4. Standard Class contributes highest delay volume due to scale.
5. Delay impact on profit exists but is moderate.

## Recommendations

- Recalibrate SLA commitments for premium shipping modes.
- Focus operational optimization efforts on Standard Class due to high volume impact.
- Align customer expectations with realistic delivery timelines.